In [30]:
import os

In [31]:
%pwd

'c:\\Users\\saqib\\OneDrive\\Desktop'

In [32]:
os.chdir("../")
%pwd

'c:\\Users\\saqib\\OneDrive'

In [20]:
from dataclasses import dataclass
from pathlib import Path

@dataclass
class DataIngestionConfig:
    root_dir: Path
    source_URL: str
    local_data_file: Path
    unzip_dir: Path

In [21]:
!pip3 install PyYAML
!pip3 install joblib
%pip install ensure
#%pip install python-box


Note: you may need to restart the kernel to use updated packages.


In [22]:
import sys
!{sys.executable} -m pip install python-box
from box import ConfigBox
print("Success! ConfigBox is ready.")

Success! ConfigBox is ready.


In [23]:
from src.productiongradewinepredictor.constants import *
from src.productiongradewinepredictor.utils.common import read_yaml, create_directories

In [24]:
class ConfigurationManager:
    def __init__(self,
                 config_filepath=CONFIG_FILE_PATH,
                 params_filepath = PARAMS_FILE_PATH,
                 schema_filepath = SCHEMA_FILE_PATH):
        self.config = read_yaml(config_filepath)
        self.params = read_yaml(params_filepath)
        self.schema = read_yaml(schema_filepath)

        create_directories([self.config.artifacts_root])


    def get_data_ingestion_config(self)-> DataIngestionConfig:
        config=self.config.data_ingestion
        create_directories([config.root_dir])

        data_ingestion_config = DataIngestionConfig(
            root_dir=config.root_dir,
            source_URL=config.source_URL,
            local_data_file=config.local_data_file,
            unzip_dir=config.unzip_dir

        )
        return data_ingestion_config

In [25]:
import os
import urllib.request as request
from src.productiongradewinepredictor import logger
import zipfile

In [26]:
## component-Data Ingestion

class DataIngestion:
    def __init__(self,config:DataIngestionConfig):
        self.config=config
    
    # Downloading the zip file
    def download_file(self):
        if not os.path.exists(self.config.local_data_file):
            filename, headers = request.urlretrieve(
                url = self.config.source_URL,
                filename = self.config.local_data_file
            )
            logger.info(f"{filename} download! with following info: \n{headers}")
        else:
            logger.info(f"File already exists")

    def extract_zip_file(self):
        """
        zip_file_path: str
        Extracts the zip file into the data directory
        Function returns None
        """
        unzip_path = self.config.unzip_dir
        os.makedirs(unzip_path, exist_ok=True)
        with zipfile.ZipFile(self.config.local_data_file, 'r') as zip_ref:
            zip_ref.extractall(unzip_path)

In [29]:
pwd

'c:\\Users\\saqib\\OneDrive\\Desktop'

In [ ]:
import os

# Set this to  actual project path
project_path = r"c:\Users\saqib\OneDrive\Desktop\production-grade-wine-predictor"

os.chdir(project_path)

print(f"Now at: {os.getcwd()}")
print(f"Does 'config' exist here? {os.path.exists('config')}")

Now at: c:\Users\saqib\OneDrive\Desktop\production-grade-wine-predictor
Does 'config' exist here? True


In [38]:
try:
    config=ConfigurationManager()
    data_ingestion_config=config.get_data_ingestion_config()
    data_ingestion=DataIngestion(config=data_ingestion_config)
    data_ingestion.download_file()
    data_ingestion.extract_zip_file()
except Exception as e:
    raise e

[2026-03-24 23:14:22,440: INFO: common: yaml file: config\config.yaml loaded successfully]
[2026-03-24 23:14:22,444: INFO: common: yaml file: params.yaml loaded successfully]
[2026-03-24 23:14:22,446: INFO: common: yaml file: schema.yaml loaded successfully]
[2026-03-24 23:14:22,447: INFO: common: created directory at: artifacts]
[2026-03-24 23:14:22,449: INFO: common: created directory at: artifacts/data_ingestion]
[2026-03-24 23:14:22,450: INFO: 535005154: File already exists]
